# Task 2: Citation Mapping - BERT Sequence Classification

**Model:** bert-base-uncased (Sequence Classification)

**Bài toán:** Cho text có [CITATION_X] + danh sách candidate papers (title + abstract)
→ Predict [CITATION_X] map với paper nào?

**Approach:** Với mỗi cặp (context quanh citation, candidate paper), model predict match (1) hoặc không match (0)

**Dataset:** task2-citation-mapping

---

## 1. Setup & Imports

In [ ]:
import transformers, datasets, accelerate
print(f"✅ transformers: {transformers.__version__}")
print(f"✅ datasets: {datasets.__version__}")
print(f"✅ accelerate: {accelerate.__version__}")

## 2. Wandb Login

In [ ]:
import wandb
from kaggle_secrets import UserSecretsClient

try:
    secrets = UserSecretsClient()
    key = secrets.get_secret("WANDB_API_KEY")
    result = wandb.login(key=key)
    if result:
        print("✅ Wandb logged in")
    else:
        print("⚠️ Wandb login returned False")
except Exception as e:
    print(f"⚠️ Wandb login failed: {type(e).__name__}: {e}")

## 3. Load & Explore Data

In [ ]:
import os

# ⚠️ Điều chỉnh path cho đúng với dataset trên Kaggle
DATA_ROOT = "/kaggle/input/datasets/tathiyennhi/task2-citation-mapping/task2"

train_path = os.path.join(DATA_ROOT, "train")
val_path = os.path.join(DATA_ROOT, "val")

# Nếu cấu trúc khác, uncomment dòng dưới để xem
# for root, dirs, files in os.walk("/kaggle/input"):
#     print(root, dirs[:5], files[:5])

train_count = len([f for f in os.listdir(train_path) if f.endswith('.label')])
val_count = len([f for f in os.listdir(val_path) if f.endswith('.label')])

print(f"✅ Train: {train_count:,} files")
print(f"✅ Val: {val_count:,} files")

In [ ]:
import json
from pathlib import Path

# Xem 1 sample .in và .label để hiểu cấu trúc data
sample_label = sorted(Path(train_path).glob("*.label"))[0]
sample_in = sample_label.with_suffix(".in")

print("=== FILE .in (input) ===")
with open(sample_in) as f:
    in_data = json.load(f)
print(f"Keys: {list(in_data.keys())}")
print(f"Text (first 200 chars): {in_data.get('text', '')[:200]}")
print(f"Num candidates: {len(in_data.get('citation_candidates', []))}")
print(f"Num bib_entries: {len(in_data.get('bib_entries', {}))}")

print("\n=== FILE .label (ground truth) ===")
with open(sample_label) as f:
    label_data = json.load(f)
print(f"Keys: {list(label_data.keys())}")
print(f"correct_citation: {label_data.get('correct_citation', {})}")

## 4. Load Data → Training Format

**Strategy:** Với mỗi citation trong document:
- Lấy context theo CONTEXT_MODE: 'full' (cả đoạn) hoặc 'window_N' (N câu xung quanh marker)
- Với mỗi candidate paper: tạo 1 training example
  - input = [CLS] context [SEP] candidate title. candidate abstract [SEP]
  - label = 1 nếu candidate đúng, 0 nếu sai

**Ablation:** Đổi CONTEXT_MODE để chạy experiments khác nhau

**Lưu ý:** Data imbalanced (1 positive vs nhiều negatives) → dùng neg_ratio để control

In [ ]:
import json
import re
import random
from pathlib import Path
from datasets import Dataset

# ═══════════════════════════════════════════════════════════
# ⚙️ CONTEXT MODE — thay đổi giá trị này để chạy ablation
# ═══════════════════════════════════════════════════════════
# 'full'     → dùng toàn bộ đoạn text (baseline, có nhiễu)
# 'window_0' → chỉ câu chứa citation marker
# 'window_1' → câu chứa + 1 câu trước/sau
# 'window_2' → câu chứa + 2 câu trước/sau
CONTEXT_MODE = 'full'
# ═══════════════════════════════════════════════════════════


def get_context(text, citation_id, mode='full'):
    """
    Trích xuất context theo mode.
    - 'full': toàn bộ text
    - 'window_N': N câu trước + câu chứa marker + N câu sau
    """
    if mode == 'full':
        return text
    
    # Parse window size từ mode string
    window = int(mode.split('_')[1])
    
    sentences = re.split(r'(?<=[.!?])\s+', text)
    
    target_idx = -1
    for i, sent in enumerate(sentences):
        if citation_id in sent:
            target_idx = i
            break
    
    if target_idx == -1:
        return text
    
    start = max(0, target_idx - window)
    end = min(len(sentences), target_idx + window + 1)
    
    return ' '.join(sentences[start:end])


def load_task2_data(data_dir, max_files=None, neg_ratio=3, context_mode='full'):
    """
    Load task 2 data → training examples.
    
    Args:
        data_dir: thư mục chứa .in và .label files
        max_files: giới hạn số files (để debug)
        neg_ratio: số negative samples cho mỗi positive
        context_mode: 'full' | 'window_0' | 'window_1' | 'window_2'
    """
    data_path = Path(data_dir)
    label_files = sorted(data_path.glob('*.label'))
    
    if max_files:
        label_files = label_files[:max_files]
    
    total_files = len(label_files)
    print(f'📊 Loading {total_files:,} files from {data_dir}')
    print(f'   Context mode: {context_mode}')
    
    examples = []
    skipped = 0
    stats = {'positive': 0, 'negative': 0}
    
    for file_idx, label_file in enumerate(label_files):
        if (file_idx + 1) % 1000 == 0:
            print(f'⏳ {file_idx+1:,}/{total_files:,} | Examples: {len(examples):,} | Skipped: {skipped}')
        
        in_file = label_file.with_suffix('.in')
        
        try:
            with open(in_file) as f:
                in_data = json.load(f)
            with open(label_file) as f:
                label_data = json.load(f)
        except:
            skipped += 1
            continue
        
        text = in_data.get('text', '')
        if not text:
            skipped += 1
            continue
        
        candidates = in_data.get('citation_candidates', [])
        bib_entries = in_data.get('bib_entries', {})
        correct_citation = label_data.get('correct_citation', {})
        
        if not correct_citation or not candidates or not bib_entries:
            skipped += 1
            continue
        
        for citation_id, correct_paper_id in correct_citation.items():
            context = get_context(text, citation_id, mode=context_mode)
            
            # Positive: correct paper
            if correct_paper_id in bib_entries:
                paper = bib_entries[correct_paper_id]
                paper_text = f"{paper.get('title', '')}. {paper.get('abstract', '')}"
                examples.append({'text_a': context, 'text_b': paper_text, 'label': 1})
                stats['positive'] += 1
            
            # Negatives: random sample from wrong candidates
            neg_candidates = [c for c in candidates if c != correct_paper_id and c in bib_entries]
            neg_sample = random.sample(neg_candidates, min(neg_ratio, len(neg_candidates)))
            
            for neg_paper_id in neg_sample:
                paper = bib_entries[neg_paper_id]
                paper_text = f"{paper.get('title', '')}. {paper.get('abstract', '')}"
                examples.append({'text_a': context, 'text_b': paper_text, 'label': 0})
                stats['negative'] += 1
    
    print(f'\n✅ Loaded {len(examples):,} examples | Skipped: {skipped}')
    print(f'   Positive: {stats["positive"]:,} | Negative: {stats["negative"]:,}')
    print(f'   Ratio: 1:{stats["negative"] / max(stats["positive"], 1):.1f}')
    
    return examples


# ── Load data ──
print('=' * 60)
print(f'CONTEXT MODE: {CONTEXT_MODE}')
print('=' * 60)

print('\nLoading TRAIN data...')
train_examples = load_task2_data(train_path, neg_ratio=3, context_mode=CONTEXT_MODE)

print('\nLoading VAL data...')
val_examples = load_task2_data(val_path, neg_ratio=3, context_mode=CONTEXT_MODE)

# Convert to HuggingFace Dataset
train_dataset = Dataset.from_list(train_examples)
val_dataset = Dataset.from_list(val_examples)

print(f'\n✅ Train dataset: {len(train_dataset):,}')
print(f'✅ Val dataset: {len(val_dataset):,}')

In [ ]:
# Kiểm tra vài samples
print("=== SAMPLE POSITIVE ===")
for ex in train_examples:
    if ex["label"] == 1:
        print(f"Context (first 200): {ex['text_a'][:200]}")
        print(f"Paper (first 200): {ex['text_b'][:200]}")
        print(f"Label: {ex['label']}")
        break

print("\n=== SAMPLE NEGATIVE ===")
for ex in train_examples:
    if ex["label"] == 0:
        print(f"Context (first 200): {ex['text_a'][:200]}")
        print(f"Paper (first 200): {ex['text_b'][:200]}")
        print(f"Label: {ex['label']}")
        break

## 5. Tokenization

In [ ]:
from transformers import AutoTokenizer

MODEL_NAME = "bert-base-uncased"
MAX_LENGTH = 512

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
print(f"✅ Tokenizer loaded: {MODEL_NAME}")


def tokenize_function(examples):
    """
    Tokenize cặp (context, paper_text) cho sequence classification.
    Input format: [CLS] context [SEP] paper_title. paper_abstract [SEP]
    """
    return tokenizer(
        examples["text_a"],
        examples["text_b"],
        max_length=MAX_LENGTH,
        truncation=True,
        padding="max_length",
    )


print("Tokenizing train dataset...")
train_tokenized = train_dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=["text_a", "text_b"],
)

print("Tokenizing val dataset...")
val_tokenized = val_dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=["text_a", "text_b"],
)

print(f"\n✅ Train tokenized: {len(train_tokenized):,}")
print(f"✅ Val tokenized: {len(val_tokenized):,}")
print(f"Columns: {train_tokenized.column_names}")

## 6. Load Model

In [ ]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2  # match (1) vs not match (0)
)
print(f"✅ Model loaded: {MODEL_NAME} (num_labels=2)")

## 7. Metrics

In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score, precision_recall_fscore_support


def compute_metrics(pred):
    """
    Compute Accuracy, Precision, Recall, F1 cho binary classification.
    """
    labels = pred.label_ids
    preds = np.argmax(pred.predictions, axis=1)
    
    accuracy = accuracy_score(labels, preds)
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, preds, average="binary", pos_label=1
    )
    
    return {
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1,
    }


print("✅ Metrics function defined")

## 8. Training Configuration

In [ ]:
from transformers import (
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
)
import wandb

WANDB_PROJECT = "task2-citation-mapping"
CHECKPOINT_DIR = "/kaggle/working/checkpoints/task2_bert"
SAVE_DIR = "/kaggle/working/task2_bert_final"

# ── Wandb init ──
try:
    if wandb.run is None:
        wandb.init(
            project=WANDB_PROJECT,
            name="bert-base-lr3e5-batch16",
            config={
                "model": MODEL_NAME,
                "task": "citation-mapping-classification",
                "learning_rate": 3e-5,
                "batch_size": 16,
                "num_epochs": 5,
                "warmup_steps": 500,
                "weight_decay": 0.01,
                "max_length": MAX_LENGTH,
                "neg_ratio": 3,
            },
            resume="allow",
        )
    report_to = "wandb"
    print("✅ Wandb initialized")
except:
    report_to = "none"
    print("⚠️ Wandb not available, logging to none")


# ── Training Arguments ──
training_args = TrainingArguments(
    output_dir=CHECKPOINT_DIR,
    
    # Training params
    num_train_epochs=5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    gradient_accumulation_steps=2,  # effective batch size = 16 * 2 = 32
    
    # Optimizer
    learning_rate=3e-5,
    weight_decay=0.01,
    warmup_steps=500,
    
    # Evaluation
    eval_strategy="steps",
    eval_steps=500,
    save_strategy="steps",
    save_steps=500,
    save_total_limit=3,
    
    # Best model
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    
    # Logging
    logging_dir="/kaggle/working/logs",
    logging_steps=100,
    report_to=report_to,
    
    # Performance
    fp16=True,
    dataloader_num_workers=2,
)


# ── Trainer ──
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tokenized,
    eval_dataset=val_tokenized,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
)

print("\n✅ Trainer configured")
print(f"   Effective batch size: {training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps}")
print(f"   Learning rate: {training_args.learning_rate}")
print(f"   Epochs: {training_args.num_train_epochs}")
print(f"   Warmup steps: {training_args.warmup_steps}")

## 9. Train

In [ ]:
print("=" * 60)
print("🚀 TRAINING BERT FOR CITATION MAPPING")
print("=" * 60)

trainer.train()

print("\n✅ Training complete!")

## 10. Evaluation

In [ ]:
print("📊 VALIDATION RESULTS")
print("=" * 60)

eval_results = trainer.evaluate()

for key, value in eval_results.items():
    if isinstance(value, float):
        print(f"{key}: {value:.4f}")
    else:
        print(f"{key}: {value}")

print("=" * 60)
print(f"\n✅ Accuracy: {eval_results.get('eval_accuracy', 0):.2%}")
print(f"✅ Precision: {eval_results.get('eval_precision', 0):.2%}")
print(f"✅ Recall: {eval_results.get('eval_recall', 0):.2%}")
print(f"✅ F1 Score: {eval_results.get('eval_f1', 0):.2%}")

## 11. Save Model

In [ ]:
import os
import shutil

os.makedirs(SAVE_DIR, exist_ok=True)

# Save model & tokenizer
print("📦 Saving final model and tokenizer...")
trainer.save_model(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)

# Zip for download
shutil.make_archive(SAVE_DIR, 'zip', SAVE_DIR)

print(f"✅ Model saved to: {SAVE_DIR}")
print(f"📦 Zip file: {SAVE_DIR}.zip")

## 12. Test Inference

Test trên 1 document: cho text + candidates → rank candidates → predict citation mapping

In [ ]:
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForSequenceClassification

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# Load saved model
test_tokenizer = AutoTokenizer.from_pretrained(SAVE_DIR)
test_model = AutoModelForSequenceClassification.from_pretrained(SAVE_DIR)
test_model.to(DEVICE)
test_model.eval()


def predict_citation_mapping(text, citation_id, candidates, bib_entries):
    """
    Cho text + citation_id + candidates → rank candidates → return best match.
    """
    context = get_context(text, citation_id, mode=CONTEXT_MODE)
    
    scores = {}
    
    for paper_id in candidates:
        if paper_id not in bib_entries:
            continue
        
        paper = bib_entries[paper_id]
        paper_text = f"{paper.get('title', '')}. {paper.get('abstract', '')}"
        
        inputs = test_tokenizer(
            context,
            paper_text,
            max_length=MAX_LENGTH,
            truncation=True,
            padding="max_length",
            return_tensors="pt"
        )
        inputs = {k: v.to(DEVICE) for k, v in inputs.items()}
        
        with torch.no_grad():
            outputs = test_model(**inputs)
            probs = F.softmax(outputs.logits, dim=1)
            match_score = probs[0][1].item()  # prob of class 1 (match)
        
        scores[paper_id] = match_score
    
    # Sort by match score descending
    ranked = sorted(scores.items(), key=lambda x: x[1], reverse=True)
    
    return ranked


# ── Test trên 1 sample từ val set ──
sample_label = sorted(Path(val_path).glob("*.label"))[0]
sample_in = sample_label.with_suffix(".in")

with open(sample_in) as f:
    in_data = json.load(f)
with open(sample_label) as f:
    label_data = json.load(f)

text = in_data["text"]
candidates = in_data["citation_candidates"]
bib_entries = in_data["bib_entries"]
correct = label_data["correct_citation"]

print("=" * 60)
print("📋 CITATION MAPPING TEST")
print("=" * 60)

for citation_id, correct_paper_id in correct.items():
    ranked = predict_citation_mapping(text, citation_id, candidates, bib_entries)
    
    predicted_id = ranked[0][0] if ranked else "N/A"
    predicted_score = ranked[0][1] if ranked else 0
    
    is_correct = predicted_id == correct_paper_id
    
    print(f"\n{citation_id}")
    print(f"  Correct:   {correct_paper_id}")
    print(f"  Predicted: {predicted_id} (score: {predicted_score:.4f})")
    print(f"  Result:    {'✅' if is_correct else '❌'}")
    
    # Show top 3 ranked
    print(f"  Top 3:")
    for pid, score in ranked[:3]:
        marker = "←" if pid == correct_paper_id else ""
        title = bib_entries.get(pid, {}).get('title', 'Unknown')[:60]
        print(f"    {score:.4f} | {pid} | {title}... {marker}")

## 13. Experiment Summary

In [ ]:
import os

summary = f"""
============================================================
       EXPERIMENT RESULTS - Task 2 Citation Mapping
============================================================

[Model Configuration]
  Model:           {MODEL_NAME}
  Max length:      {MAX_LENGTH}
  Learning rate:   {training_args.learning_rate}
  Batch size:      {training_args.per_device_train_batch_size} x {training_args.gradient_accumulation_steps} grad accum = {training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps}
  Epochs:          {training_args.num_train_epochs}
  Warmup steps:    {training_args.warmup_steps}
  Weight decay:    {training_args.weight_decay}
  FP16:            {training_args.fp16}

[Data]
  Train examples:  {len(train_tokenized):,}
  Val examples:    {len(val_tokenized):,}
  Neg ratio:       3

[Validation Metrics]
  Accuracy:        {eval_results.get('eval_accuracy', 0):.4f}
  Precision:       {eval_results.get('eval_precision', 0):.4f}
  Recall:          {eval_results.get('eval_recall', 0):.4f}
  F1:              {eval_results.get('eval_f1', 0):.4f}
  Eval loss:       {eval_results.get('eval_loss', 0):.4f}

[WandB]
  Project:         {WANDB_PROJECT}

============================================================
"""

print(summary)

# Save report
output_dir = "/kaggle/working/results"
os.makedirs(output_dir, exist_ok=True)
with open(f"{output_dir}/task2_report.txt", "w") as f:
    f.write(summary)
print(f"✅ Report saved to {output_dir}/task2_report.txt")

# Finish wandb
try:
    wandb.finish()
except:
    pass

print("\n🎉 TASK 2 TRAINING COMPLETE!")